Baseline Pipeline

In [1]:
import time
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

In [4]:
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, confusion_matrix

def run_baseline_pipeline(data_path):
    df = pd.read_csv(data_path)
    df_filtered = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
    
    # Add ALL THREE engineered features (match self-healing)
    df_filtered['balance_diff_orig'] = (
        df_filtered['oldbalanceOrg'] - df_filtered['amount'] - df_filtered['newbalanceOrig']
    )
    df_filtered['orig_zero_after'] = (
        (df_filtered['oldbalanceOrg'] > 0) & (df_filtered['newbalanceOrig'] == 0)
    ).astype(int)
    df_filtered['dest_no_change'] = (
        (df_filtered['amount'] > 0) & 
        (df_filtered['oldbalanceDest'] == df_filtered['newbalanceDest'])
    ).astype(int)
    
    # SAME 8 features as self-healing
    feature_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 
                    'oldbalanceDest', 'newbalanceDest', 'balance_diff_orig',
                    'orig_zero_after', 'dest_no_change']
    
    X_baseline = df_filtered[feature_cols].fillna(0)
    y_baseline = df_filtered['isFraud']
    
    contamination_rate = y_baseline.mean()
    
    start_time = time.time()
    model = IsolationForest(
        n_estimators=100,
        contamination=contamination_rate,
        random_state=42,
        n_jobs=-1
    )
    preds = model.fit_predict(X_baseline)
    end_time = time.time()
    
    preds_binary = (preds == -1).astype(int)
    
    delta_t = end_time - start_time
    precision = precision_score(y_baseline, preds_binary, pos_label=1, zero_division=0)
    recall = recall_score(y_baseline, preds_binary, pos_label=1, zero_division=0)
    f1 = f1_score(y_baseline, preds_binary, pos_label=1, zero_division=0)
    cm = confusion_matrix(y_baseline, preds_binary)
    
    print(f"Execution Latency: {delta_t:.4f} seconds")
    print(f"Confusion Matrix:\n{cm}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    
    return {
        "data_path": data_path,
        "delta_t": delta_t,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

if __name__ == "__main__":
    baseline_results = run_baseline_pipeline('Dataset.csv')

Execution Latency: 58.0780 seconds
Confusion Matrix:
[[2754284    7912]
 [   7921     292]]
Precision: 0.0356
Recall: 0.0356
F1-Score: 0.0356


In [5]:
run_baseline_pipeline('paysim_noise_5.csv')
run_baseline_pipeline('paysim_noise_10.csv')
run_baseline_pipeline('paysim_noise_20.csv')

Execution Latency: 50.3420 seconds
Confusion Matrix:
[[2754692    7504]
 [   7511     702]]
Precision: 0.0855
Recall: 0.0855
F1-Score: 0.0855
Execution Latency: 51.7772 seconds
Confusion Matrix:
[[2754326    7870]
 [   7870     343]]
Precision: 0.0418
Recall: 0.0418
F1-Score: 0.0418
Execution Latency: 49.7061 seconds
Confusion Matrix:
[[2754330    7866]
 [   7866     347]]
Precision: 0.0423
Recall: 0.0423
F1-Score: 0.0423


{'data_path': 'paysim_noise_20.csv',
 'delta_t': 49.70612359046936,
 'precision': 0.04225009131864118,
 'recall': 0.04225009131864118,
 'f1_score': 0.04225009131864118}

In [7]:
def run_baseline_streaming_pipeline(file_path, chunk_size=100000, 
                                     retrain_interval=10):
    """Streaming baseline with periodic retraining (no validation, no repair)."""
    print(f"\n{'='*60}")
    print(f"STREAMING BASELINE (Periodic Retrain) ON: {file_path}")
    print(f"{'='*60}")

    start_time = time.time()
    
    y_true_all = []
    y_pred_all = []
    
    scaler = None
    model = None
    refit_count = 0
    chunk_count = 0
    
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        chunk_count += 1
        
        # Filter to fraud-relevant transaction types
        df_filtered = chunk[chunk['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
        if df_filtered.empty:
            continue
        
        # 1. Feature Engineering (on FILTERED data)
        df_filtered['balance_diff_orig'] = (
            df_filtered['oldbalanceOrg'] - df_filtered['amount'] - df_filtered['newbalanceOrig']
        )
        df_filtered['orig_zero_after'] = (
            (df_filtered['oldbalanceOrg'] > 0) & (df_filtered['newbalanceOrig'] == 0)
        ).astype(int)
        df_filtered['dest_no_change'] = (
            (df_filtered['amount'] > 0) & 
            (df_filtered['oldbalanceDest'] == df_filtered['newbalanceDest'])
        ).astype(int)
        
        feature_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig',
                        'oldbalanceDest', 'newbalanceDest',
                        'balance_diff_orig', 'orig_zero_after', 'dest_no_change']
        
        X = df_filtered[feature_cols].fillna(0)
        y = df_filtered['isFraud'].values
        
        # 2. Periodic Retraining
        if model is None or (chunk_count - 1) % retrain_interval == 0:
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)
            
            model = IsolationForest(
                n_estimators=100,
                contamination=max(y.mean(), 0.001),
                random_state=42,
                n_jobs=-1
            )
            model.fit(X_scaled)
            refit_count += 1
            preds = model.predict(X_scaled)
            print(f"[REFIT] Batch {chunk_count} (refit #{refit_count}), "
                  f"contamination={max(y.mean(), 0.001):.4f}, "
                  f"filtered_rows={len(df_filtered)}")
        else:
            X_scaled = scaler.transform(X)
            preds = model.predict(X_scaled)
        
        binary_preds = (preds == -1).astype(int)
        y_true_all.extend(y)
        y_pred_all.extend(binary_preds)
    
    total_latency = time.time() - start_time
    
    precision = precision_score(y_true_all, y_pred_all, zero_division=0)
    recall = recall_score(y_true_all, y_pred_all, zero_division=0)
    f1 = f1_score(y_true_all, y_pred_all, zero_division=0)
    
    print(f"\n{'='*40}")
    print(f"STREAMING BASELINE RESULTS")
    print(f"{'='*40}")
    print(f"Batches: {chunk_count}, Refits: {refit_count}")
    print(f"Latency: {total_latency:.4f}s")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1:        {f1:.4f}")
    
    return {'f1': f1, 'precision': precision, 'recall': recall, 
            'latency': total_latency, 'refits': refit_count}

In [8]:
if __name__ == "__main__":
    for f in ['Dataset.csv', 'paysim_noise_5.csv', 
              'paysim_noise_10.csv', 'paysim_noise_20.csv']:
        run_baseline_streaming_pipeline(f)


STREAMING BASELINE (Periodic Retrain) ON: Dataset.csv
[REFIT] Batch 1 (refit #1), contamination=0.0030, filtered_rows=39315
[REFIT] Batch 11 (refit #2), contamination=0.0280, filtered_rows=34047
[REFIT] Batch 21 (refit #3), contamination=0.0010, filtered_rows=45703
[REFIT] Batch 31 (refit #4), contamination=0.0010, filtered_rows=46514
[REFIT] Batch 41 (refit #5), contamination=0.0010, filtered_rows=45024
[REFIT] Batch 51 (refit #6), contamination=0.0010, filtered_rows=45952
[REFIT] Batch 61 (refit #7), contamination=0.0241, filtered_rows=41243

STREAMING BASELINE RESULTS
Batches: 64, Refits: 7
Latency: 25.4618s
Precision: 0.1277
Recall:    0.1953
F1:        0.1544

STREAMING BASELINE (Periodic Retrain) ON: paysim_noise_5.csv
[REFIT] Batch 1 (refit #1), contamination=0.0016, filtered_rows=100000
[REFIT] Batch 11 (refit #2), contamination=0.0015, filtered_rows=100000
[REFIT] Batch 21 (refit #3), contamination=0.0017, filtered_rows=100000

STREAMING BASELINE RESULTS
Batches: 28, Refits: 